In [ ]:
import os
import datetime
import pandas as pd
import numpy as np
from lxml import etree
from tqdm import tqdm

In [ ]:
# The base_path should be the results with labels generated in ../data_preparations/add_labels.py
base_path = r"C:\Users\Olive\python_projects\FinRL-Meta\FinGPT-ChatGLM-Fineturning_old\Chinese_News\scrape\content_with_labels"
columns = ["instruction","input","output"]
instruction = "What is the sentiment of this news? Answer:{very negative/negative/neutral/positive/very positive}"

file_list = os.listdir(base_path)
print(instruction)

### Title/标题

In [ ]:
dataset = pd.DataFrame(columns= columns)
for file_name in tqdm(file_list):
    df = pd.read_csv(os.path.join(base_path, file_name))
    df["input"] = df.apply(lambda x: f'今天的日期为：{x["publish_date"]}。\n新闻标题为：\"{x["post_title"]}\"。', axis = 1)
    df["instruction"] = instruction
    df['output'] = df["label"]
    tmp = df[columns+["date"]]
    dataset = pd.concat([dataset, tmp])

In [ ]:
dataset

In [ ]:
dataset = dataset[dataset.output != "No data"]

In [ ]:
dataset.date.sort_values().unique()

In [ ]:
dataset.groupby("date").date.count().plot()

#####  Test from 2022-04-01 till 2023-03-31
#####  Train & valid from 2011-05-12 till 2022-03-31

In [ ]:
dataset[dataset.date<"2022-04-01"].shape

In [ ]:
dataset.shape

In [ ]:
print(df["input"][0])

In [ ]:
# Train & valid
train_valid = dataset[dataset.date < "2022-04-01"][columns]
train_valid.to_csv("content/dataset_title_train_and_valid.csv", index = False)

In [ ]:
# Test
test = dataset[dataset.date >= "2022-04-01"][columns]
test.to_csv("content/dataset_title_test.csv", index = False)

In [ ]:
data_list = []
for item in train_valid.itertuples():
    tmp = {}
    tmp["instruction"] = item.instruction
    tmp["input"] = item.input
    tmp["output"] = item.output
    data_list.append(tmp)

In [ ]:
# To Json
import json
with open("content/dataset_title_train_and_valid.json", "w+", encoding='utf-8') as f:
    json.dump(data_list, f, ensure_ascii= False)

In [ ]:
# To Json
import json
with open("content/dataset_title_test.json", "w+", encoding='utf-8') as f:
    json.dump(data_list, f, ensure_ascii= False)

### Contetn/内容

In [ ]:
def get_content(x):
    try:
        tree = etree.HTML(x)
        content = tree.xpath("//*[name(.)!='style']/text()")
        content = "".join(content)
        content = content.replace("\u3000", '')
        return content
    except:
        return np.nan

In [ ]:
dataset_content = pd.DataFrame(columns= columns)
for file_name in tqdm(file_list):
    df = pd.read_csv(os.path.join(base_path, file_name))  
    df['input'] = df['post_content'].apply(get_content)
    df = df.dropna(subset=['input'])
    df["input"] = df.apply(lambda x: f'今天的日期为：{x["publish_date"]}。\n新闻内容为：\"{x["input"]}\"。', axis = 1)
    df["instruction"] = instruction
    df['output'] = df["label"]
    tmp = df[columns+["date"]]
    dataset_content = pd.concat([dataset_content, tmp])

In [ ]:
print(df["input"][0])

In [ ]:
dataset_content = dataset_content[dataset_content.output != "No data"]
dataset_content.shape

In [ ]:
dataset_content["total_len"] = dataset_content["input"].apply(lambda x:len(x.split(" ")))
dataset_content = dataset_content[dataset_content["total_len"] < 1000]
dataset_content.shape

In [ ]:
dataset_content["total_len"].max()

In [ ]:
# Train & valid
train_valid = dataset_content[dataset_content.date < "2022-04-01"][columns]
train_valid.to_csv("content/dataset_content_train_and_valid.csv", index = False)

In [ ]:
# Test
test = dataset_content[dataset_content.date >= "2022-04-01"][columns]
test.to_csv("content/dataset_content_test.csv", index = False)

In [ ]:
data_list = []
for item in train_valid.itertuples():
    tmp = {}
    tmp["instruction"] = item.instruction
    tmp["input"] = item.input
    tmp["output"] = item.output
    data_list.append(tmp)

In [ ]:
import json
with open("content/dataset_content_train_and_valid.json", "w+", encoding='utf-8') as f:
    json.dump(data_list, f, ensure_ascii= False)

In [ ]:
dataset_content[dataset_content.date < "2022-04-01"].groupby("output").count()

In [ ]:
dataset_content[dataset_content.date < "2022-04-01"].output.hist()